In [ ]:
from os.path import sep

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("vendas.csv", sep=";")

In [ ]:
#1. Exploração Inicial dos dados
print("--- Informações do DataFrame ---")
display(df.info())

print("\n--- Resumo Estatístico de Variáveis Numéricas ---")
display(df.describe())

print("\n--- Primeiras 5 linhas do DataFrame ---")
display(df.head())

In [ ]:
#2. Tratamento de dados para identificar valores ausentes, duplicados e demais inconsistencias.

# Verificar o tamanho da tabela
linhas, colunas = df.shape
print(f"Quantidade de registros (linhas): {linhas}")
print(f"Quantidade de variáveis (colunas): {colunas}")

# Verificação de valores nulos
print("\n--- Valores Nulos por Coluna ---")
nulos = df.isnull().sum()
display(nulos[nulos > 0] if nulos.sum() > 0 else "Nenhum valor nulo encontrado!")

# Anomalias numéricas encontradas (valores negativos em quantidade e total)
print("\n--- Inconsistências de Valores Negativos ---")
print(f"Registros com quantidade negativa: {len(df[df['quantidade'] < 0])}")
print(f"Registros com valor total negativo: {len(df[df['valor_total'] < 0])}")

In [ ]:
# Criando uma cópia para preservar os dados originais
df_clean = df.copy()

# Removendo linhas com valores nulos na coluna 'cidade_cliente'
df_clean = df_clean.dropna(subset=['cidade_cliente'])

# Filtrando inconsistências lógicas (valores maiores que zero)
df_clean = df_clean[(df_clean['quantidade'] > 0) & (df_clean['valor_total'] > 0)]

# Verificando e removendo registros duplicados
duplicados_antes = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()

# Convertendo a coluna 'data_venda' para datetime (coagindo erros em NaT e removendo-os)
df_clean['data_venda'] = pd.to_datetime(df_clean['data_venda'], errors='coerce')
linhas_data_invalida = df_clean['data_venda'].isnull().sum()
df_clean = df_clean.dropna(subset=['data_venda'])

# Verificação final pós-limpeza
print("--- Resultados do Tratamento de Dados ---")
print(f"Linhas duplicadas removidas: {duplicados_antes}")
print(f"Linhas com datas corrompidas removidas: {linhas_data_invalida}")
print(f"Registros restantes na base tratada: {df_clean.shape[0]} (Redução de {df.shape[0] - df_clean.shape[0]} linhas no total)")
print(f"Valores nulos restantes: {df_clean.isnull().sum().sum()}")
print(f"Menor quantidade na base: {df_clean['quantidade'].min()}")
print(f"Menor valor total na base: {df_clean['valor_total'].min()}")
print(f"Tipo de dado da coluna data_venda: {df_clean['data_venda'].dtype}")

In [ ]:

# 3. Faturamento  exibe o total do faturamento bruto e o total do faturamento liquido.

print("Status de venda disponíveis:", df_clean['status_venda'].unique())

# Cálculo do Faturamento Bruto (soma do valor_total de todos os itens de pedidos não cancelados/gerais)
faturamento_bruto_total = df_clean['valor_total'].sum()

# Cálculo do Faturamento Líquido
# Consideramos apenas vendas com status 'Concluído' (ou equivalente) para o faturamento real líquido
vendas_concluidas = df_clean[df_clean['status_venda'] == 'Concluído']
faturamento_liquido = vendas_concluidas['valor_total'].sum()

print("\n--- Resultados Financeiros ---")
print(f"Faturamento Bruto Total (Geral): R$ {faturamento_bruto_total:,.2f}")
print(f"Faturamento Líquido (Apenas Vendas Concluídas): R$ {faturamento_liquido:,.2f}")

In [ ]:
# 4.Top 10 produtos por quantidade de transações
top_10_transacoes = df_clean['nome_produto'].value_counts().head(10).reset_index()
top_10_transacoes.columns = ['Nome do Produto', 'Frequência de Vendas']

# Top 10 produtos por volume total de itens vendidos
top_10_quantidade = df_clean.groupby('nome_produto')['quantidade'].sum().sort_values(ascending=False).head(10).reset_index()
top_10_quantidade.columns = ['Nome do Produto', 'Total de Itens Vendidos']
# Imprimi os 10 produtos mais vendidos por transação.
print("--- TOP 10 PRODUTOS MAIS VENDIDOS (POR TRANSAÇÕES) ---")
display(top_10_transacoes)
#Imprimi os 10 produtos mais vendidos por volume de itens.
print("\n--- TOP 10 PRODUTOS MAIS VENDIDOS (POR VOLUME DE ITENS) ---")
display(top_10_quantidade)

In [ ]:
# 5. Desempenho por categoria identifica as categorias de produtos que geraram o maior volume financeiro para a empresa.

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Limpeza e tratamento
df_clean = df.copy()
df_clean = df_clean.dropna(subset=['cidade_cliente'])
df_clean = df_clean[(df_clean['quantidade'] > 0) & (df_clean['valor_total'] > 0)]
df_clean = df_clean.drop_duplicates()
df_clean['data_venda'] = pd.to_datetime(df_clean['data_venda'], errors='coerce')
df_clean = df_clean.dropna(subset=['data_venda'])

# Filtrar apenas as vendas concluídas
vendas_concluidas = df_clean[df_clean['status_venda'] == 'Concluído']

# Agrupar por categoria de produto
faturamento_por_categoria = (
    vendas_concluidas.groupby('categoria_produto', as_index=False)['valor_total']
    .sum()
    .sort_values(by='valor_total', ascending=False)
)
faturamento_por_categoria.columns = ['Categoria', 'Faturamento Líquido (R$)']

# Gerar a visualização corrigida
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(12, 6))

sns.barplot(
    x='Faturamento Líquido (R$)',
    y='Categoria',
    hue='Categoria',
    data=faturamento_por_categoria,
    palette='viridis',
    legend=False,
    ax=ax
)

# Garante margem à direita para os rótulos não serem cortados
max_val = faturamento_por_categoria['Faturamento Líquido (R$)'].max()
ax.set_xlim(0, max_val * 1.15)

# Adiciona os rótulos formatados em Reais (BR)
for index, value in enumerate(faturamento_por_categoria['Faturamento Líquido (R$)']):
    valor_br = f" R$ {value:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    ax.text(value, index, valor_br, va='center', fontsize=10, fontweight='bold')

ax.set_title('Faturamento Líquido por Categoria de Produto (Vendas Concluídas)', fontsize=14, pad=15)
ax.set_xlabel('Faturamento Líquido (R$)', fontsize=12)
ax.set_ylabel('Categoria', fontsize=12)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

In [ ]:
# 6. Identifica o maior e menor periodo de faturamento
import pandas as pd

try:

    df_clean
except NameError:
    df = pd.read_csv("vendas.csv", sep=";")
    df_clean = df.copy()
    df_clean = df_clean.dropna(subset=['cidade_cliente'])
    df_clean = df_clean[(df_clean['quantidade'] > 0) & (df_clean['valor_total'] > 0)]
    df_clean = df_clean.drop_duplicates()
    df_clean['data_venda'] = pd.to_datetime(df_clean['data_venda'], errors='coerce')
    df_clean = df_clean.dropna(subset=['data_venda'])

# Filtrar apenas as vendas concluídas
vendas_concluidas = df_clean[df_clean['status_venda'] == 'Concluído']

# Definir o período (Ano-Mês) nas vendas concluídas
vendas_concluidas = vendas_concluidas.copy()
vendas_concluidas['ano_mes'] = vendas_concluidas['data_venda'].dt.to_period('M')

# Agrupar por 'ano_mes' e somar o 'valor_total'
evolucao_temporal = vendas_concluidas.groupby('ano_mes')['valor_total'].sum()

# Identificar os meses de maior e menor faturamento
mes_maior_faturamento = evolucao_temporal.idxmax()
valor_maximo = evolucao_temporal.max()

mes_menor_faturamento = evolucao_temporal.idxmin()
valor_minimo = evolucao_temporal.min()

# Agora, execute esta célula para descobrir os resultados salvos nas variáveis acima!
print(f"Maior Faturamento: {mes_maior_faturamento} -> R$ {valor_maximo:,.2f}")
print(f"Menor Faturamento: {mes_menor_faturamento} -> R$ {valor_minimo:,.2f}")



In [ ]:
# 7. Avaliação de equipe identifica os 5 melhores vendedores com desempenho financeiro.

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

#Carregamento dos dados necessarios
try:

    df_temp = df_clean.copy()
except NameError:

    df_temp = pd.read_csv("vendas.csv", sep=";")
    df_temp = df_temp.dropna(subset=['cidade_cliente'])
    df_temp = df_temp[(df_temp['quantidade'] > 0) & (df_temp['valor_total'] > 0)]
    df_temp = df_temp.drop_duplicates()

# Filtrar apenas as vendas concluídas para a avaliação financeira
vendas_concluidas_temp = df_temp[df_temp['status_venda'] == 'Concluído']

# Agrupar as vendas concluídas por vendedor
desempenho_vendedores = vendas_concluidas_temp.groupby('nome_vendedor').agg(
    faturamento_total=('valor_total', 'sum'),
    quantidade_pedidos=('id_pedido', 'nunique'),
    ticket_medio=('valor_total', 'mean')
).reset_index()

# Identificar os 5 melhores por faturamento
top_5_vendedores = desempenho_vendedores.sort_values(by='faturamento_total', ascending=False).head(5)

# Exibir a tabela formatada
print("--- TOP 5 VENDEDORES: DESEMPENHO E TICKET MÉDIO ---")
display(top_5_vendedores.style.format({
    'faturamento_total': 'R$ {:,.2f}',
    'ticket_medio': 'R$ {:,.2f}'
}).hide(axis='index'))

# Gerar gráfico comparativo de faturamento
plt.figure(figsize=(10, 5))
sns.set_theme(style="whitegrid")
ax = sns.barplot(
    x='faturamento_total',
    y='nome_vendedor',
    data=top_5_vendedores,
    hue='nome_vendedor',
    palette='Blues_r',
    legend=False
)

# Adicionar os rótulos de valor nas barras
for index, value in enumerate(top_5_vendedores['faturamento_total']):
    ax.text(value, index, f' R$ {value:,.2f}', va='center', fontsize=10, fontweight='bold')

plt.title('Top 5 Vendedores por Faturamento Líquido (Vendas Concluídas)', fontsize=14, pad=15)
plt.xlabel('Faturamento Líquido (R$)', fontsize=12)
plt.ylabel('Vendedor', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
#8. Analisa as formas de pagamento utilizadas pelos clientes.
status_venda_col = 'status_venda'
forma_pagamento_col = 'forma_pagamento'

if status_venda_col in df.columns and forma_pagamento_col in df.columns:
    # Identificar quais são os status de transações concluídas/sucesso (ex: 'Concluído', 'Pago', 'Entregue')
    # Vamos analisar os status presentes
    print("Status de venda únicos:", df[status_venda_col].unique())

    # Filtrando as transações concluídas.
    concluidas = df[df[status_venda_col].str.lower().str.strip().isin(['concluido', 'concluído', 'pago', 'entregue'])]


    if concluidas.empty:
        concluidas = df[~df[status_venda_col].str.lower().str.strip().isin(['cancelado', 'devolvido'])]

    # Calcular a contagem e percentual
    payment_counts = concluidas[forma_pagamento_col].value_counts()
    payment_percentages = concluidas[forma_pagamento_col].value_counts(normalize=True) * 100

    summary_df = pd.DataFrame({
        'Transações Concluídas': payment_counts,
        'Participação (%)': payment_percentages
    })
    display(summary_df)
else:
    print("Colunas necessárias não encontradas no DataFrame.")

In [ ]:
# 9.Compara o desempenho comercial entre os estados presentes na base.
geo_analise = concluidas.groupby('estado_cliente').agg(
    total_faturamento=('valor_total', 'sum'),
    quantidade_itens=('quantidade', 'sum'),
    total_pedidos=('id_pedido', 'nunique')
).reset_index()

# Calculando a participação percentual no faturamento total
total_geral_faturamento = geo_analise['total_faturamento'].sum()
geo_analise['participacao_faturamento_pct'] = (geo_analise['total_faturamento'] / total_geral_faturamento) * 100

# Ordenando pelos estados com maior faturamento
geo_analise = geo_analise.sort_values(by='total_faturamento', ascending=False).reset_index(drop=True)

display(geo_analise)

In [ ]:
#10. Visualização de dados


import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(3, 2, figsize=(16, 20))
fig.suptitle('Visualizações Analíticas do Desempenho de Vendas', fontsize=20, fontweight='bold', y=0.98)

# Gráfico de Pizza - Participação das Formas de Pagamento
axes[0, 0].pie(payment_percentages, labels=payment_percentages.index, autopct='%1.2f%%', startangle=140,
            colors=sns.color_palette('pastel')[0:4], wedgeprops={'edgecolor': 'gray', 'linewidth': 0.8})
axes[0, 0].set_title('1. Participação Percentual das Formas de Pagamento', fontsize=12, fontweight='bold')

# Gráfico de Barras - Quantidade de Transações por Pagamento
sns.barplot(x=payment_counts.index, y=payment_counts.values, ax=axes[0, 1], palette='Blues_r', hue=payment_counts.index, legend=False)
axes[0, 1].set_title('2. Total de Transações Concluídas por Método', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Número de Transações')
axes[0, 1].set_xlabel('Forma de Pagamento')
for p in axes[0, 1].patches:
    axes[0, 1].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')

# Gráfico de Barras - Faturamento por Estado (UF)
sns.barplot(x='estado_cliente', y='total_faturamento', data=geo_analise, ax=axes[1, 0], palette='viridis', hue='estado_cliente', legend=False)
axes[1, 0].set_title('3. Faturamento Acumulado por Estado (R$)', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Faturamento (R$) em Milhões')
axes[1, 0].set_xlabel('Estado (UF)')

# Gráfico de Linha/Marcadores - Pedidos Únicos por Estado
sns.lineplot(x='estado_cliente', y='total_pedidos', data=geo_analise, ax=axes[1, 1], marker='o', color='crimson', linewidth=2.5, markersize=8)
axes[1, 1].set_title('4. Total de Pedidos Únicos por Estado (UF)', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Pedidos Únicos')
axes[1, 1].set_xlabel('Estado (UF)')

# Gráfico de Barras - Quantidade de Itens Vendidos por Estado
sns.barplot(x='estado_cliente', y='quantidade_itens', data=geo_analise, ax=axes[2, 0], palette='rocket', hue='estado_cliente', legend=False)
axes[2, 0].set_title('5. Quantidade de Itens Vendidos por Estado (UF)', fontsize=12, fontweight='bold')
axes[2, 0].set_ylabel('Total de Itens')
axes[2, 0].set_xlabel('Estado (UF)')

# Gráfico de Dispersão com Regressão - Faturamento vs Quantidade de Itens
sns.regplot(x='quantidade_itens', y='total_faturamento', data=geo_analise, ax=axes[2, 1], color='darkorange', scatter_kws={'s': 100})
axes[2, 1].set_title('6. Correlação: Faturamento vs Quantidade de Itens', fontsize=12, fontweight='bold')
axes[2, 1].set_ylabel('Faturamento Total (R$)')
axes[2, 1].set_xlabel('Quantidade de Itens Vendidos')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# Investigação estratégica: Otimização de Marketing e vendas por Estado e Categoria

#1. Preparação dos Dados e Cruzamento de Dimensões
#Realizei o cruzamento das variaveis de Faturamento por Estado, Categoria de Produto e analisei o impacto das perdas financeiras por Vendas Canceladas/Devolvidas

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Cruzamento de Dimensões: Faturamento por Estado e Categoria de Produto
estado_categoria = concluidas.groupby(['estado_cliente', 'categoria_produto']).agg(
    faturamento_liquido=('valor_total', 'sum'),
    quantidade_vendida=('quantidade', 'sum')
).reset_index()

# Pivotar para melhor visualização estruturada
pivot_est_cat = estado_categoria.pivot(index='estado_cliente', columns='categoria_produto', values='faturamento_liquido').fillna(0)
pivot_est_cat['Faturamento_Total'] = pivot_est_cat.sum(axis=1)
pivot_est_cat = pivot_est_cat.sort_values(by='Faturamento_Total', ascending=False)

print("--- FATURAMENTO LÍQUIDO POR ESTADO E CATEGORIA DE PRODUTO ---")
display(pivot_est_cat.style.format("R$ {:,.2f}"))


In [ ]:
#2. Analise de Gargalo: Taxa de Cancelamento e Faturamento Perdido por Estado
#Para otimizar o orçamento, não basta olhar para onde se vende mais, mas também onde estamos gastando energia com vendas que não se concretizam.

# Calcular faturamento total pretendido (incluindo cancelados e devolvidos)
df['is_perda'] = df['status_venda'].str.lower().str.strip().isin(['cancelado', 'devolvido'])

perda_geo = df.groupby(['estado_cliente', 'is_perda']).agg(
    valor_total=('valor_total', 'sum')
).unstack(fill_value=0).reset_index()

perda_geo.columns = ['estado_cliente', 'faturamento_concluido', 'faturamento_perdido']
perda_geo['faturamento_potencial'] = perda_geo['faturamento_concluido'] + perda_geo['faturamento_perdido']
perda_geo['taxa_perda_pct'] = (perda_geo['faturamento_perdido'] / perda_geo['faturamento_potencial']) * 100
perda_geo = perda_geo.sort_values(by='taxa_perda_pct', ascending=False).reset_index(drop=True)

print("--- TAXA DE PERDA FINANCEIRA (CANCELADOS/DEVOLVIDOS) POR ESTADO ---")
display(perda_geo.style.format({
    'faturamento_concluido': "R$ {:,.2f}",
    'faturamento_perdido': "R$ {:,.2f}",
    'faturamento_potencial': "R$ {:,.2f}",
    'taxa_perda_pct': "{:.2f}%"
}))

In [ ]:
#3. Visualização Gráfica
#Vamos gerar gráficos para sustentar visualmente a tomada de decisão para a diretoria.

# Criando figura com duas visualizações estratégicas
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Gráfico 1: Heatmap de Faturamento Estado x Categoria
sns.heatmap(pivot_est_cat.drop(columns='Faturamento_Total').head(5),
            annot=True, fmt=".2g", cmap="YlGnBu", ax=axes[0])
axes[0].set_title("Top 5 Estados: Faturamento por Categoria (R$)", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Categoria de Produto", fontsize=12)
axes[0].set_ylabel("Estado do Cliente", fontsize=12)

# Gráfico 2: Taxa de Perda de Faturamento por Estado
sns.barplot(x='estado_cliente', y='taxa_perda_pct', data=perda_geo,
            palette='OrRd_r', ax=axes[1], hue='estado_cliente', legend=False)
axes[1].axhline(df['is_perda'].mean() * 100, color='red', linestyle='--', label=f'Média de Perda Global ({df["is_perda"].mean()*100:.2f}%)')
axes[1].set_title("Taxa de Perda Financeira (%) por Estado (Cancelados/Devolvidos)", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Estado do Cliente", fontsize=12)
axes[1].set_ylabel("Taxa de Perda (%)", fontsize=12)
axes[1].legend()

plt.tight_layout()
plt.show()

4. Conclusão Executiva e Recomendação Estratégica.

Onde Investir: Os dados demostram que estados do Sul e Sudeste como PR, SC e SP concentram o maior faturamento absoluto em categorias chave. Recomenda-se alocar 60% do orçamento de marketing especificamente para campanhas digitais geolocalizadas focadas nestas regiões de altissima conversão.
Embora apresentem alto volume, estados com taxas de cancelamento acima da média necessitam de uma intervenção na força de vendas ou no fluxo de checkout de pagamento antes de receberem novos aportes de marketing. Sugere-se uma força-tarefa de pós-venda para entender gargalos